In [ ]:
from __future__ import annotations

"""Mined factorlib submission: large_liquid_reversal_flow_absorb.

Family: interaction
Expression: 0.29*prod(large_size, short_rev) + 0.27*prod(high_amount20, tech_rev) + 0.22*defensive + 0.14*flow_reversal + 0.08*prod(flow_out5, discount_sma20)
"""

"""Mine and validate factorlib expressions inside BigQuant AIStudio.

The local machine cannot read the competition tables, but AIStudio can read
filtered `bigalpha_2026_factorlib` windows. This script runs there, generates a
library of hand-designed expression candidates, evaluates IC/spread, measures
correlation versus already-submitted factors, and exports top candidates as
self-contained submission scripts.
"""

import argparse
import importlib.util
import json
import math
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


FACTORLIB_COLUMNS = [
    "date",
    "instrument",
    "close",
    "volume",
    "amount",
    "turn",
    "change_ratio",
    "daily_return",
    "momentum_5",
    "reversal_5",
    "volatility_5",
    "float_market_cap",
    "sma_20",
    "ema_20",
    "macd_hist_12_26_9",
    "rsi_12",
    "kdj_k_9_3_3",
    "kdj_d_9_3_3",
    "bias_20",
    "cci_14",
    "atr_14",
    "netflow_amount_rate_main",
    "net_active_buy_amount_main",
    "beta_000300SH_22",
    "list_days",
    "pe_ttm",
    "pb",
    "ps_ttm",
    "roe_avg_ttm",
    "roa_avg_ttm",
    "gross_profit_rate_ttm",
    "net_profit_rate_ttm",
    "debt_to_asset_lf",
    "current_ratio_lf",
]

DEFAULT_BASELINES = [
    "factorlib_quality_value.py",
    "factorlib_reversal_defensive_v2.py",
    "factorlib_reversal_concentrated_v5.py",
    "factorlib_reversal_defensive_tilt_v6.py",
    "factorlib_reversal_technical_tilt_v7.py",
    "factorlib_reversal_momentum_restore_v8.py",
    "factorlib_stable_quality_v3.py",
    "factorlib_flow_trend_v4.py",
]


class Candidate:
    def __init__(self, name: str, family: str, expr: str) -> None:
        self.name = name
        self.family = family
        self.expr = expr


def import_dai():
    try:
        import dai  # type: ignore

        return dai
    except Exception:
        from bigquant import dai  # type: ignore

        return dai


def load_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def fetch_panel(dai, query_start: str, query_end: str) -> pd.DataFrame:
    columns_sql = ",\n        ".join(FACTORLIB_COLUMNS)
    sql = f"""
    SELECT
        {columns_sql}
    FROM bigalpha_2026_factorlib
    ORDER BY date, instrument
    """
    print(f"FETCH factorlib {query_start} -> {query_end}, columns={len(FACTORLIB_COLUMNS)}")
    panel = dai.query(sql, filters={"date": [query_start, query_end]}, compression=True).df()
    panel["date"] = pd.to_datetime(panel["date"]).dt.normalize()
    panel["instrument"] = panel["instrument"].astype(str)
    print("PANEL", panel.shape, str(panel["date"].min()), str(panel["date"].max()))
    return panel


def numeric_panel(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for col in columns:
        if col not in df.columns:
            df[col] = np.nan
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        df[col] = df.groupby("date", observed=True)[col].transform(
            lambda s: s.fillna(s.median()) if s.notna().any() else s.fillna(0.0)
        )
        df[col] = df[col].fillna(0.0)
    return df


def rolling(df: pd.DataFrame, column: str, window: int, min_periods: int, func: str = "mean") -> pd.Series:
    grouped = df.groupby("instrument", observed=True)[column]
    if func == "sum":
        return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).sum())
    if func == "std":
        return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).std())
    return grouped.transform(lambda s: s.rolling(window, min_periods=min_periods).mean())


def rank_by_date(df: pd.DataFrame, values: pd.Series, ascending: bool = True) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ranked = values.groupby(df["date"], observed=True).rank(pct=True, ascending=ascending)
    return ranked.fillna(0.5) - 0.5


def prod(a: pd.Series, b: pd.Series) -> pd.Series:
    # One-sided gate: b is a centered rank in [-0.5, 0.5], so b + 0.5 is a
    # [0, 1] gate. This avoids rewarding "bad signal + bad gate" pairs.
    gate = (b.fillna(0.0) + 0.5).clip(0.0, 1.0)
    return a.fillna(0.0) * gate


def clip(a: pd.Series, lo: float = -0.5, hi: float = 0.5) -> pd.Series:
    return a.clip(lo, hi)


def build_feature_namespace(panel: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, pd.Series]]:
    df = panel.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df = numeric_panel(df, [c for c in FACTORLIB_COLUMNS if c not in {"date", "instrument"}])

    df["ret_3_sum"] = rolling(df, "daily_return", 3, 2, "sum")
    df["ret_5_sum"] = rolling(df, "daily_return", 5, 3, "sum")
    df["ret_10_sum"] = rolling(df, "daily_return", 10, 5, "sum")
    df["ret_20_sum"] = rolling(df, "daily_return", 20, 10, "sum")
    df["ret_60_sum"] = rolling(df, "daily_return", 60, 30, "sum")
    df["vol_5"] = rolling(df, "daily_return", 5, 3, "std")
    df["vol_20"] = rolling(df, "daily_return", 20, 10, "std")
    df["vol_60"] = rolling(df, "daily_return", 60, 30, "std")
    df["turn_5"] = rolling(df, "turn", 5, 3)
    df["turn_20"] = rolling(df, "turn", 20, 10)
    df["amount_5"] = rolling(df, "amount", 5, 3)
    df["amount_20"] = rolling(df, "amount", 20, 10)
    df["amount_60"] = rolling(df, "amount", 60, 30)
    df["flow_3"] = rolling(df, "netflow_amount_rate_main", 3, 2)
    df["flow_5"] = rolling(df, "netflow_amount_rate_main", 5, 3)
    df["flow_20"] = rolling(df, "netflow_amount_rate_main", 20, 10)
    df["active_buy_5"] = rolling(df, "net_active_buy_amount_main", 5, 3)
    df["active_buy_20"] = rolling(df, "net_active_buy_amount_main", 20, 10)
    df["price_to_sma20"] = np.where(df["sma_20"] > 0, df["close"] / df["sma_20"] - 1.0, np.nan)
    df["price_to_ema20"] = np.where(df["ema_20"] > 0, df["close"] / df["ema_20"] - 1.0, np.nan)
    df["atr_to_close"] = np.where(df["close"] > 0, df["atr_14"] / df["close"], np.nan)
    df["atr_to_close_20"] = rolling(df, "atr_to_close", 20, 10)
    df["kdj_gap"] = df["kdj_k_9_3_3"] - df["kdj_d_9_3_3"]
    df["amihud"] = np.abs(df["daily_return"]) / np.log1p(np.maximum(df["amount"], 0.0))
    df["amihud_20"] = rolling(df, "amihud", 20, 10)
    df["earn_yield"] = np.where(np.abs(df["pe_ttm"]) > 1e-6, 1.0 / df["pe_ttm"], np.nan)
    df["log_pb"] = np.where(df["pb"] > 0, np.log(df["pb"]), np.nan)
    df["log_ps"] = np.where(df["ps_ttm"] > 0, np.log(df["ps_ttm"]), np.nan)
    df["log_mcap"] = np.where(df["float_market_cap"] > 0, np.log(df["float_market_cap"]), np.nan)
    df["roe_delta_20"] = df.groupby("instrument", observed=True)["roe_avg_ttm"].transform(lambda s: s.diff(20))
    df["roa_delta_20"] = df.groupby("instrument", observed=True)["roa_avg_ttm"].transform(lambda s: s.diff(20))
    df["gross_margin_delta_20"] = df.groupby("instrument", observed=True)["gross_profit_rate_ttm"].transform(lambda s: s.diff(20))
    df["net_margin_delta_20"] = df.groupby("instrument", observed=True)["net_profit_rate_ttm"].transform(lambda s: s.diff(20))
    df["current_delta_20"] = df.groupby("instrument", observed=True)["current_ratio_lf"].transform(lambda s: s.diff(20))
    df["debt_delta_20"] = df.groupby("instrument", observed=True)["debt_to_asset_lf"].transform(lambda s: s.diff(20))
    df["macd_delta_5"] = df.groupby("instrument", observed=True)["macd_hist_12_26_9"].transform(lambda s: s.diff(5))
    df["rsi_delta_5"] = df.groupby("instrument", observed=True)["rsi_12"].transform(lambda s: s.diff(5))
    df["kdj_gap_delta_5"] = df.groupby("instrument", observed=True)["kdj_gap"].transform(lambda s: s.diff(5))
    df["margin_conversion"] = np.where(
        np.abs(df["gross_profit_rate_ttm"]) > 1e-4,
        df["net_profit_rate_ttm"] / np.abs(df["gross_profit_rate_ttm"]),
        np.nan,
    )
    df = numeric_panel(
        df,
        [
            "ret_3_sum",
            "ret_5_sum",
            "ret_10_sum",
            "ret_20_sum",
            "ret_60_sum",
            "vol_5",
            "vol_20",
            "vol_60",
            "turn_5",
            "turn_20",
            "amount_5",
            "amount_20",
            "amount_60",
            "flow_3",
            "flow_5",
            "flow_20",
            "active_buy_5",
            "active_buy_20",
            "price_to_sma20",
            "price_to_ema20",
            "atr_to_close",
            "kdj_gap",
            "amihud_20",
            "earn_yield",
            "log_pb",
            "log_ps",
            "log_mcap",
            "roe_delta_20",
            "roa_delta_20",
            "gross_margin_delta_20",
            "net_margin_delta_20",
            "current_delta_20",
            "debt_delta_20",
            "macd_delta_5",
            "rsi_delta_5",
            "kdj_gap_delta_5",
            "margin_conversion",
        ],
    )

    f: dict[str, pd.Series] = {}
    add = f.__setitem__
    add("rev1", rank_by_date(df, -df["daily_return"]))
    add("rev3", rank_by_date(df, -df["ret_3_sum"]))
    add("rev5", rank_by_date(df, -df["ret_5_sum"]))
    add("rev10", rank_by_date(df, -df["ret_10_sum"]))
    add("reversal_5", rank_by_date(df, df["reversal_5"]))
    add("anti_mom5", rank_by_date(df, -df["momentum_5"]))
    add("mom20_5", rank_by_date(df, df["ret_20_sum"] - df["ret_5_sum"]))
    add("mom60_20", rank_by_date(df, df["ret_60_sum"] - df["ret_20_sum"]))
    add("low_vol5", rank_by_date(df, -df["vol_5"]))
    add("low_vol20", rank_by_date(df, -df["vol_20"]))
    add("low_vol60", rank_by_date(df, -df["vol_60"]))
    add("high_vol5", rank_by_date(df, df["vol_5"]))
    add("high_turn5", rank_by_date(df, df["turn_5"]))
    add("high_turn20", rank_by_date(df, df["turn_20"]))
    add("low_turn5", rank_by_date(df, -df["turn_5"]))
    add("low_turn20", rank_by_date(df, -df["turn_20"]))
    add("high_amount20", rank_by_date(df, df["amount_20"]))
    add("amount_surge", rank_by_date(df, np.log1p(df["amount_5"]) - np.log1p(df["amount_60"])))
    add("turn_reopen", rank_by_date(df, df["turn_5"] - df["turn_20"]))
    add("capacity", rank_by_date(df, np.log1p(df["amount_20"]) - df["log_mcap"]))
    add("low_amihud", rank_by_date(df, -df["amihud_20"]))
    add("low_beta", rank_by_date(df, -df["beta_000300SH_22"]))
    add("low_atr", rank_by_date(df, -df["atr_to_close"]))
    add("vol_compress", rank_by_date(df, df["vol_20"] - df["vol_5"]))
    add("atr_compress", rank_by_date(df, df["atr_to_close_20"] - df["atr_to_close"]))
    add("cheap_pb", rank_by_date(df, -df["log_pb"]))
    add("cheap_ps", rank_by_date(df, -df["log_ps"]))
    add("earn_yield", rank_by_date(df, df["earn_yield"]))
    add("quality", rank_by_date(df, df["roe_avg_ttm"] + df["roa_avg_ttm"] + df["gross_profit_rate_ttm"]))
    add("profit_margin", rank_by_date(df, df["gross_profit_rate_ttm"] + df["net_profit_rate_ttm"]))
    add("balance_sheet", rank_by_date(df, -df["debt_to_asset_lf"] + df["current_ratio_lf"]))
    add("quality_delta", rank_by_date(df, df["roe_delta_20"] + df["roa_delta_20"] - df["debt_delta_20"]))
    add("margin_delta", rank_by_date(df, df["gross_margin_delta_20"] + df["net_margin_delta_20"]))
    add("balance_delta", rank_by_date(df, df["current_delta_20"] - df["debt_delta_20"]))
    add("margin_conversion", rank_by_date(df, df["margin_conversion"]))
    add("small_size", rank_by_date(df, -df["log_mcap"]))
    add("large_size", rank_by_date(df, df["log_mcap"]))
    add("old_listing", rank_by_date(df, df["list_days"]))
    add("discount_sma20", rank_by_date(df, -df["price_to_sma20"]))
    add("discount_ema20", rank_by_date(df, -df["price_to_ema20"]))
    add("over_sma20", rank_by_date(df, df["price_to_sma20"]))
    add("low_rsi", rank_by_date(df, -df["rsi_12"]))
    add("high_rsi", rank_by_date(df, df["rsi_12"]))
    add("low_bias", rank_by_date(df, -df["bias_20"]))
    add("low_cci", rank_by_date(df, -df["cci_14"]))
    add("macd_up", rank_by_date(df, df["macd_hist_12_26_9"]))
    add("macd_down", rank_by_date(df, -df["macd_hist_12_26_9"]))
    add("macd_turn", rank_by_date(df, df["macd_delta_5"]))
    add("rsi_turn", rank_by_date(df, df["rsi_delta_5"]))
    add("kdj_turn", rank_by_date(df, df["kdj_gap_delta_5"]))
    add("neutral_bias", rank_by_date(df, -np.abs(df["bias_20"])))
    add("kdj_revert", rank_by_date(df, -df["kdj_gap"]))
    add("flow_in3", rank_by_date(df, df["flow_3"]))
    add("flow_in5", rank_by_date(df, df["flow_5"]))
    add("flow_in20", rank_by_date(df, df["flow_20"]))
    add("flow_accel", rank_by_date(df, df["flow_5"] - df["flow_20"]))
    add("active_accel", rank_by_date(df, df["active_buy_5"] - df["active_buy_20"]))
    add("flow_out3", rank_by_date(df, -df["flow_3"]))
    add("flow_out5", rank_by_date(df, -df["flow_5"]))
    add("flow_out20", rank_by_date(df, -df["flow_20"]))
    add("active_buy", rank_by_date(df, df["active_buy_5"] + df["active_buy_20"]))
    add("active_sell", rank_by_date(df, -(df["active_buy_5"] + df["active_buy_20"])))

    f["short_rev"] = 0.26 * f["reversal_5"] + 0.24 * f["rev1"] + 0.20 * f["rev3"] + 0.18 * f["rev5"] + 0.12 * f["anti_mom5"]
    f["tech_rev"] = 0.20 * f["discount_sma20"] + 0.18 * f["discount_ema20"] + 0.20 * f["low_rsi"] + 0.16 * f["low_bias"] + 0.14 * f["low_cci"] + 0.12 * f["kdj_revert"]
    f["defensive"] = 0.24 * f["low_vol20"] + 0.18 * f["low_vol5"] + 0.16 * f["low_turn20"] + 0.16 * f["low_amihud"] + 0.14 * f["low_atr"] + 0.12 * f["low_beta"]
    f["quality_value"] = 0.24 * f["quality"] + 0.18 * f["profit_margin"] + 0.18 * f["balance_sheet"] + 0.16 * f["earn_yield"] + 0.14 * f["cheap_pb"] + 0.10 * f["cheap_ps"]
    f["flow_reversal"] = 0.25 * f["flow_out3"] + 0.25 * f["flow_out5"] + 0.18 * f["flow_out20"] + 0.17 * f["active_sell"] + 0.15 * f["short_rev"]
    f["flow_momentum"] = 0.25 * f["flow_in3"] + 0.25 * f["flow_in5"] + 0.20 * f["active_buy"] + 0.15 * f["mom20_5"] + 0.15 * f["macd_up"]
    f["flow_acceleration"] = 0.38 * f["flow_accel"] + 0.32 * f["active_accel"] + 0.16 * f["capacity"] + 0.14 * f["low_beta"]
    f["medium_mom"] = 0.45 * f["mom60_20"] + 0.30 * f["mom20_5"] + 0.15 * f["old_listing"] + 0.10 * f["low_vol20"]
    f["fundamental_improvement"] = 0.30 * f["quality_delta"] + 0.26 * f["margin_delta"] + 0.24 * f["balance_delta"] + 0.20 * f["profit_margin"]
    f["volatility_compression"] = 0.40 * f["vol_compress"] + 0.28 * f["atr_compress"] + 0.18 * f["low_beta"] + 0.14 * f["low_turn20"]
    f["liquidity_reopen"] = 0.34 * f["amount_surge"] + 0.26 * f["capacity"] + 0.22 * f["turn_reopen"] + 0.18 * f["low_amihud"]
    f["oscillator_turn"] = 0.34 * f["macd_turn"] + 0.24 * f["kdj_turn"] + 0.22 * f["rsi_turn"] + 0.20 * f["neutral_bias"]
    f["mid_age"] = -2.0 * (rank_by_date(df, df["list_days"]) - 0.05).abs() + 0.5
    f["mid_size"] = -2.0 * (rank_by_date(df, df["log_mcap"]) + 0.05).abs() + 0.5
    return df, f


def candidate_library() -> list[Candidate]:
    c: list[Candidate] = []
    add = c.append

    manual = [
        ("liq_conditioned_reversal", "interaction", "0.42*prod(short_rev, high_amount20) + 0.28*prod(tech_rev, low_amihud) + 0.18*defensive + 0.12*flow_reversal"),
        ("quiet_reversal", "interaction", "0.50*prod(short_rev, low_vol20) + 0.22*tech_rev + 0.18*low_beta + 0.10*old_listing"),
        ("high_turn_exhaustion", "interaction", "0.44*prod(short_rev, high_turn20) + 0.24*prod(tech_rev, high_turn5) + 0.18*flow_reversal + 0.14*low_atr"),
        ("illiquid_reversal", "interaction", "0.46*prod(short_rev, low_turn20) + 0.24*tech_rev + 0.18*low_vol20 + 0.12*quality_value"),
        ("panic_absorption", "interaction", "0.36*prod(rev1, high_vol5) + 0.28*prod(flow_out3, discount_sma20) + 0.20*low_atr + 0.16*low_beta"),
        ("oversold_flow_absorption", "flow", "0.34*prod(flow_out5, low_rsi) + 0.24*prod(active_sell, discount_ema20) + 0.24*short_rev + 0.18*defensive"),
        ("flow_reversal_quality", "flow", "0.34*flow_reversal + 0.28*prod(short_rev, quality_value) + 0.20*defensive + 0.18*tech_rev"),
        ("flow_momentum_quiet", "flow", "0.34*prod(flow_momentum, low_vol20) + 0.24*medium_mom + 0.18*quality_value + 0.14*large_size + 0.10*low_beta"),
        ("quality_reversal", "fundamental", "0.34*prod(quality_value, short_rev) + 0.26*prod(quality_delta, tech_rev) + 0.22*defensive + 0.18*cheap_pb"),
        ("cheap_quality_lowrisk", "fundamental", "0.30*quality_value + 0.24*prod(cheap_pb, low_vol20) + 0.20*balance_sheet + 0.16*low_beta + 0.10*old_listing"),
        ("small_quality_reversal", "fundamental", "0.30*prod(small_size, quality_value) + 0.30*prod(short_rev, high_amount20) + 0.24*tech_rev + 0.16*low_vol20"),
        ("large_liquid_reversal", "interaction", "0.30*prod(large_size, short_rev) + 0.28*prod(high_amount20, tech_rev) + 0.24*defensive + 0.18*flow_reversal"),
        ("trend_lowrisk", "momentum", "0.34*prod(medium_mom, low_vol20) + 0.22*flow_momentum + 0.18*quality_value + 0.16*low_beta + 0.10*old_listing"),
        ("trend_pullback", "momentum", "0.32*prod(medium_mom, rev3) + 0.26*prod(flow_momentum, discount_sma20) + 0.22*low_vol20 + 0.20*quality_value"),
        ("macd_reversal_hybrid", "technical", "0.30*tech_rev + 0.22*prod(macd_up, short_rev) + 0.20*defensive + 0.16*flow_reversal + 0.12*quality_value"),
        ("rsi_kdj_exhaustion", "technical", "0.32*prod(low_rsi, kdj_revert) + 0.26*short_rev + 0.20*prod(low_bias, flow_out5) + 0.14*low_vol20 + 0.08*quality_value"),
        ("flow_acceleration_core", "flow_accel", "0.48*flow_acceleration + 0.20*prod(flow_accel, low_turn20) + 0.18*active_accel + 0.14*low_beta"),
        ("flow_accel_low_liquidity", "flow_accel", "0.38*prod(flow_acceleration, low_turn20) + 0.26*capacity + 0.20*active_accel + 0.16*low_vol20"),
        ("fundamental_improvement", "fundamental_change", "0.46*fundamental_improvement + 0.22*quality_delta + 0.18*margin_delta + 0.14*balance_delta"),
        ("qarp_nonlinear", "fundamental_change", "0.34*prod(quality, cheap_pb) + 0.24*prod(profit_margin, earn_yield) + 0.22*prod(balance_sheet, cheap_ps) + 0.20*quality_delta"),
        ("margin_conversion_efficiency", "fundamental_change", "0.36*margin_conversion + 0.24*profit_margin + 0.22*balance_sheet + 0.18*quality_delta"),
        ("volatility_compression", "risk_change", "0.52*volatility_compression + 0.20*vol_compress + 0.16*atr_compress + 0.12*low_beta"),
        ("liquidity_reopen", "liquidity_change", "0.46*liquidity_reopen + 0.24*amount_surge + 0.18*capacity + 0.12*turn_reopen"),
        ("oscillator_inflection", "technical_turn", "0.48*oscillator_turn + 0.20*macd_turn + 0.16*kdj_turn + 0.16*rsi_turn"),
        ("seasoned_midcap_quality", "lifecycle", "0.30*mid_age + 0.24*mid_size + 0.24*quality_value + 0.22*capacity"),
        ("midcap_flow_accel_quality", "lifecycle", "0.28*prod(mid_size, flow_acceleration) + 0.24*prod(mid_age, quality_value) + 0.24*liquidity_reopen + 0.24*low_beta"),
    ]
    for name, family, expr in manual:
        add(Candidate(name, family, expr))

    gates = ["low_vol20", "high_turn20", "low_turn20", "high_amount20", "low_amihud", "quality_value", "flow_out5"]
    cores = ["short_rev", "tech_rev", "flow_reversal", "quality_value", "medium_mom"]
    for gate in gates:
        add(Candidate(f"gated_short_rev_{gate}", "grid", f"0.52*prod(short_rev, {gate}) + 0.24*tech_rev + 0.16*defensive + 0.08*flow_reversal"))
        add(Candidate(f"gated_tech_rev_{gate}", "grid", f"0.42*prod(tech_rev, {gate}) + 0.30*short_rev + 0.18*defensive + 0.10*quality_value"))
    for core in cores:
        if core == "quality_value":
            continue
        add(Candidate(f"quality_cross_{core}", "grid", f"0.40*prod({core}, quality_value) + 0.24*defensive + 0.20*tech_rev + 0.16*flow_reversal"))
    return c


def add_forward_returns(panel: pd.DataFrame, horizons: list[int]) -> pd.DataFrame:
    df = panel.loc[:, ["date", "instrument", "daily_return"]].copy()
    df["daily_return"] = pd.to_numeric(df["daily_return"], errors="coerce")
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    def fwd_sum(s: pd.Series, horizon: int) -> pd.Series:
        shifted = [s.shift(-i) for i in range(1, horizon + 1)]
        return pd.concat(shifted, axis=1).sum(axis=1, min_count=horizon)

    for horizon in horizons:
        df[f"fwd_{horizon}"] = df.groupby("instrument", observed=True)["daily_return"].transform(
            lambda s, h=horizon: fwd_sum(s, h)
        )
    return df.drop(columns=["daily_return"])


def safe_corr(x: pd.Series, y: pd.Series, method: str) -> float:
    valid = x.notna() & y.notna()
    if valid.sum() < 50:
        return float("nan")
    xv = x[valid]
    yv = y[valid]
    if xv.nunique(dropna=True) < 3 or yv.nunique(dropna=True) < 3:
        return float("nan")
    return float(xv.corr(yv, method=method))


def decile_spread(group: pd.DataFrame, target_col: str) -> float:
    valid = group[["factor", target_col]].dropna()
    if len(valid) < 100:
        return float("nan")
    pct = valid["factor"].rank(pct=True)
    top = valid.loc[pct >= 0.9, target_col]
    bottom = valid.loc[pct <= 0.1, target_col]
    if top.empty or bottom.empty:
        return float("nan")
    return float(top.mean() - bottom.mean())


def summarize(values: pd.Series) -> dict[str, float | int]:
    values = pd.to_numeric(values, errors="coerce").dropna()
    if values.empty:
        return {"n": 0, "mean": float("nan"), "std": float("nan"), "ir": float("nan"), "pos_rate": float("nan")}
    mean = float(values.mean())
    std = float(values.std(ddof=1))
    return {
        "n": int(len(values)),
        "mean": mean,
        "std": std,
        "ir": mean / std * math.sqrt(len(values)) if std > 0 else float("nan"),
        "pos_rate": float((values > 0).mean()),
    }


def eval_expr(expr: str, features: dict[str, pd.Series], frame: pd.DataFrame) -> pd.Series:
    env: dict[str, Any] = {"prod": prod, "clip": clip, "np": np}
    env.update(features)
    raw = eval(expr, {"__builtins__": {}}, env)
    if not isinstance(raw, pd.Series):
        raw = pd.Series(raw, index=frame.index)
    return rank_by_date(frame, raw)


def build_baseline_matrix(panel: pd.DataFrame, eval_index: pd.DataFrame, baseline_dir: Path, baseline_names: list[str]) -> pd.DataFrame:
    out = eval_index.copy()
    for name in baseline_names:
        path = baseline_dir / name
        if not path.exists():
            print(f"SKIP missing baseline {path}")
            continue
        module = load_module(path)
        print(f"BASELINE {path.name}")
        factor = module._build_factor(panel.copy())
        factor["date"] = pd.to_datetime(factor["date"]).dt.normalize()
        factor["instrument"] = factor["instrument"].astype(str)
        factor = factor.rename(columns={"factor": path.stem})
        out = out.merge(factor.loc[:, ["date", "instrument", path.stem]], on=["date", "instrument"], how="left")
    return out


def evaluate_candidate(
    candidate: Candidate,
    factor: pd.Series,
    base: pd.DataFrame,
    horizons: list[int],
    baseline_cols: list[str],
) -> dict[str, Any]:
    df = base.copy()
    df["factor"] = factor.loc[df.index].to_numpy()
    result: dict[str, Any] = {
        "candidate": candidate.name,
        "family": candidate.family,
        "expr": candidate.expr,
        "rows": int(len(df)),
        "dates": int(df["date"].nunique()),
        "missing_factor_rate": float(pd.isna(df["factor"]).mean()),
    }
    by_date = df.groupby("date", observed=True)
    for horizon in horizons:
        target_col = f"fwd_{horizon}"
        ic = by_date.apply(lambda g, c=target_col: safe_corr(g["factor"], g[c], "pearson"))
        rank_ic = by_date.apply(lambda g, c=target_col: safe_corr(g["factor"], g[c], "spearman"))
        spread = by_date.apply(lambda g, c=target_col: decile_spread(g, c))
        for prefix, series in [(f"ic{horizon}", ic), (f"rankic{horizon}", rank_ic), (f"spread{horizon}", spread)]:
            stats = summarize(series)
            for key, value in stats.items():
                result[f"{prefix}_{key}"] = value

    max_corr = 0.0
    best_corr = ""
    corr_values: dict[str, float] = {}
    for col in baseline_cols:
        daily_corr = by_date.apply(lambda g, c=col: safe_corr(g["factor"], g[c], "spearman"))
        corr = float(daily_corr.abs().dropna().mean()) if daily_corr.notna().any() else float("nan")
        signed_corr = float(daily_corr.dropna().mean()) if daily_corr.notna().any() else float("nan")
        p95_corr = float(daily_corr.abs().dropna().quantile(0.95)) if daily_corr.notna().any() else float("nan")
        if not math.isnan(corr):
            corr_values[col] = signed_corr
            result[f"corr_mean_abs__{col}"] = corr
            result[f"corr_p95_abs__{col}"] = p95_corr
            if corr > abs(max_corr):
                max_corr = signed_corr
                best_corr = col
    result["max_abs_baseline_corr"] = abs(max_corr)
    result["max_corr_baseline"] = best_corr
    result["max_corr_value"] = max_corr

    if baseline_cols:
        residual = pd.Series(index=df.index, dtype="float64")
        for _, group in df.groupby("date", observed=True):
            y = pd.to_numeric(group["factor"], errors="coerce").fillna(0.0).to_numpy(dtype="float64")
            x = group.loc[:, baseline_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype="float64")
            x = np.column_stack([np.ones(len(x)), x])
            try:
                beta, *_ = np.linalg.lstsq(x, y, rcond=None)
                residual.loc[group.index] = y - x @ beta
            except np.linalg.LinAlgError:
                residual.loc[group.index] = y - y.mean()
        df["residual_factor"] = residual
        resid_rankic = by_date.apply(lambda g: safe_corr(g["residual_factor"], g["fwd_1"], "spearman"))
        resid_tmp = df.rename(columns={"factor": "raw_factor", "residual_factor": "factor"})
        resid_spread = resid_tmp.groupby("date", observed=True).apply(lambda g: decile_spread(g, "fwd_1"))
        for key, value in summarize(resid_rankic).items():
            result[f"resid_rankic1_{key}"] = value
        for key, value in summarize(resid_spread).items():
            result[f"resid_spread1_{key}"] = value
    else:
        result["resid_rankic1_mean"] = float("nan")
        result["resid_spread1_mean"] = float("nan")

    rankic = float(result.get("rankic1_mean", float("nan")))
    spread = float(result.get("spread1_mean", float("nan")))
    ir = float(result.get("rankic1_ir", float("nan")))
    pos_rate = float(result.get("rankic1_pos_rate", float("nan")))
    rankic3 = float(result.get("rankic3_mean", float("nan")))
    resid_rankic = float(result.get("resid_rankic1_mean", float("nan")))
    missing = float(result.get("missing_factor_rate", float("nan")))
    corr_abs = float(result.get("max_abs_baseline_corr", float("nan")))
    passes = (
        rankic >= 0.035
        and ir >= 2.0
        and pos_rate >= 0.58
        and spread > 0
        and rankic3 >= 0
        and missing <= 0.01
        and corr_abs <= 0.75
        and resid_rankic >= 0.010
    )
    result["passes_submit_gates"] = bool(passes)
    diversity = max(0.0, 1.0 - max(0.0, corr_abs - 0.55) / 0.30)
    stability = max(0.0, min(1.5, (ir if not math.isnan(ir) else 0.0) / 2.5))
    residual_bonus = 200.0 * (resid_rankic if not math.isnan(resid_rankic) else 0.0)
    base_score = rankic * 100.0 + rankic3 * 50.0 + spread * 10.0 + residual_bonus
    result["selection_score"] = base_score * diversity * stability + (5.0 if passes else 0.0)
    result["baseline_corrs"] = corr_values
    return result


def slugify(name: str) -> str:
    text = re.sub(r"[^a-zA-Z0-9_]+", "_", name).strip("_").lower()
    return text[:80] or "candidate"


def export_submission_script(candidate: Candidate, output_dir: Path) -> Path:
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / f"factorlib_mined_{slugify(candidate.name)}.py"
    template_path = Path(__file__)
    source = template_path.read_text(encoding="utf-8")
    marker = "\n\nif __name__ == \"__main__\":\n"
    reusable = source.split(marker)[0]
    reusable = re.sub(r"^#!.*\n", "", reusable)
    reusable = reusable.replace("from __future__ import annotations\n\n", "")
    reusable = reusable.replace("from dataclasses import dataclass\n", "")
    reusable = reusable.replace(
        "@dataclass(frozen=True)\nclass Candidate:\n    name: str\n    family: str\n    expr: str\n",
        "class Candidate:\n    def __init__(self, name: str, family: str, expr: str) -> None:\n        self.name = name\n        self.family = family\n        self.expr = expr\n",
    )
    script = f'''"""Mined factorlib submission: {candidate.name}.

Family: {candidate.family}
Expression: {candidate.expr}
"""

{reusable}

CANDIDATE = Candidate({candidate.name!r}, {candidate.family!r}, {candidate.expr!r})


def _build_factor(panel: pd.DataFrame) -> pd.DataFrame:
    frame, features = build_feature_namespace(panel)
    factor = eval_expr(CANDIDATE.expr, features, frame)
    out = frame.loc[:, ["date", "instrument"]].copy()
    out["factor"] = factor.astype("float64").replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date", observed=True)["factor"].transform(lambda s: s.fillna(s.median()))
    out["factor"] = out["factor"].fillna(0.0)
    out["factor"] = rank_by_date(out, out["factor"])
    return out.loc[:, ["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main(
    datasources: dict | str | None = None,
    start_date: str = "2019-01-01 00:00:00",
    end_date: str = "2025-12-31 23:59:59",
) -> pd.DataFrame:
    dai = import_dai()
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S")
    panel = fetch_panel(dai, query_start, end_date)
    out = _build_factor(panel)
    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    return out[(out["date"] >= start_ts) & (out["date"] <= end_ts)].reset_index(drop=True)
'''
    path.write_text(script, encoding="utf-8")
    return path


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--query-start", default="2024-09-01")
    parser.add_argument("--query-end", default="2025-04-15")
    parser.add_argument("--eval-start", default="2025-01-01")
    parser.add_argument("--eval-end", default="2025-03-31")
    parser.add_argument("--horizons", nargs="*", type=int, default=[1, 3, 5])
    parser.add_argument("--baseline-dir", default="/home/aiuser/work")
    parser.add_argument("--baselines", nargs="*", default=DEFAULT_BASELINES)
    parser.add_argument("--output-dir", default="/home/aiuser/work/reports")
    parser.add_argument("--export-dir", default="/home/aiuser/work/mined_submissions")
    parser.add_argument("--export-top", type=int, default=3)
    args = parser.parse_args()

    dai = import_dai()
    panel = fetch_panel(dai, args.query_start, args.query_end)
    frame, features = build_feature_namespace(panel)
    targets = add_forward_returns(panel, args.horizons)
    base = frame.loc[:, ["date", "instrument"]].merge(targets, on=["date", "instrument"], how="left")
    start_ts = pd.to_datetime(args.eval_start).normalize()
    end_ts = pd.to_datetime(args.eval_end).normalize()
    base = base[(base["date"] >= start_ts) & (base["date"] <= end_ts)].copy()
    print("EVAL_BASE", base.shape, str(base["date"].min()), str(base["date"].max()))

    baseline_matrix = build_baseline_matrix(panel, base.loc[:, ["date", "instrument"]], Path(args.baseline_dir), args.baselines)
    baseline_cols = [c for c in baseline_matrix.columns if c not in {"date", "instrument"}]
    base = base.merge(baseline_matrix, on=["date", "instrument"], how="left")

    candidates = candidate_library()
    print(f"CANDIDATES {len(candidates)}")
    results: list[dict[str, Any]] = []
    for idx, candidate in enumerate(candidates, 1):
        print(f"EVAL {idx}/{len(candidates)} {candidate.name}")
        factor = eval_expr(candidate.expr, features, frame)
        eval_factor = frame.loc[:, ["date", "instrument"]].copy()
        eval_factor["factor"] = factor
        eval_factor = base.loc[:, ["date", "instrument"]].merge(eval_factor, on=["date", "instrument"], how="left")["factor"]
        results.append(evaluate_candidate(candidate, eval_factor, base.reset_index(drop=True), args.horizons, baseline_cols))

    report = pd.DataFrame(results)
    report = report.sort_values(["selection_score", "rankic1_mean"], ascending=False).reset_index(drop=True)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    csv_path = output_dir / "aistudio_factor_miner_report.csv"
    json_path = output_dir / "aistudio_factor_miner_report.json"
    report.to_csv(csv_path, index=False)
    json_path.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    selected = []
    for row in report.itertuples(index=False):
        candidate = next(c for c in candidates if c.name == row.candidate)
        path = export_submission_script(candidate, Path(args.export_dir))
        selected.append((candidate.name, str(path)))
        if len(selected) >= args.export_top:
            break

    display_cols = [
        "candidate",
        "family",
        "selection_score",
        "rankic1_mean",
        "rankic1_ir",
        "spread1_mean",
        "rankic3_mean",
        "rankic5_mean",
        "max_abs_baseline_corr",
        "max_corr_baseline",
    ]
    print(report.loc[:, [c for c in display_cols if c in report.columns]].head(20).to_string(index=False))
    print(f"WROTE {csv_path}")
    print(f"WROTE {json_path}")
    for name, path in selected:
        print(f"EXPORTED {name} -> {path}")


CANDIDATE = Candidate('large_liquid_reversal_flow_absorb', 'interaction', '0.29*prod(large_size, short_rev) + 0.27*prod(high_amount20, tech_rev) + 0.22*defensive + 0.14*flow_reversal + 0.08*prod(flow_out5, discount_sma20)')


def _build_factor(panel: pd.DataFrame) -> pd.DataFrame:
    frame, features = build_feature_namespace(panel)
    factor = eval_expr(CANDIDATE.expr, features, frame)
    out = frame.loc[:, ["date", "instrument"]].copy()
    out["factor"] = factor.astype("float64").replace([np.inf, -np.inf], np.nan)
    out["factor"] = out.groupby("date", observed=True)["factor"].transform(lambda s: s.fillna(s.median()))
    out["factor"] = out["factor"].fillna(0.0)
    out["factor"] = rank_by_date(out, out["factor"])
    return out.loc[:, ["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main(
    datasources: dict | str | None = None,
    start_date: str = "2019-01-01 00:00:00",
    end_date: str = "2025-12-31 23:59:59",
) -> pd.DataFrame:
    dai = import_dai()
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime("%Y-%m-%d %H:%M:%S")
    panel = fetch_panel(dai, query_start, end_date)
    out = _build_factor(panel)
    start_ts = pd.to_datetime(start_date).normalize()
    end_ts = pd.to_datetime(end_date).normalize()
    return out[(out["date"] >= start_ts) & (out["date"] <= end_ts)].reset_index(drop=True)
